In [ ]:
import os, json, time, requests, pandas as pd
from typing import Dict, Any, Optional
from openai import OpenAI
import os
from IPython.display import display, Markdown
import os, json, time, requests, pandas as pd


### CVE and Vulnerability Analyzer
Made this for devs, SREs, and anyone else who needs concise information on a list of CVEs. The notebook takes a list of CVEs and outputs a summary report for each. This version calls OpenAI and needs a key for that API. Place the key and the CVE numbers you want researched in the next cell. The system prompt is in the cell after that below for easy editing. 

In [ ]:

# Configure paramaters here

# ✅ CVE list for processing
CVE_LIST = [
    "CVE-2025-4008",
    "CVE-2025-25231",
    # Add more as needed
]

# ✅ Model selection (assumes you have access to gpt-5)
MODEL_NAME = "gpt-5"

# ✅ OpenAI API key — OPTION A: Define directly here (not recommended for shared notebooks)
OPENAI_API_KEY = ""

# ✅ If API key is NOT defined here, fallback to environment variable
if OPENAI_API_KEY:
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY  # sets runtime variable for client use
print("✅ Parameters loaded. CVEs:", CVE_LIST)


In [ ]:
# ---- SYSTEM PROMPT FOR OPENAI ----

DEV_JSON_INSTRUCTIONS = """
Return ONLY valid JSON with:
{
  "cve": "CVE-YYYY-NNNN",
  "severity_emoji": "🔥|⚠️|🧊",
  "affected_product": "vendor/product or 'unknown'",
  "affected_versions": "version/range or 'unknown'",
  "affected_service": "component/module or 'unknown'",
  "exploit_description": "how an attacker exploits, short and concrete",
  "impact": "result (RCE, data theft, PE, DoS, lateral movement, etc.)",
  "mitigations": ["actionable mitigation 1", "actionable mitigation 2"],
  "patch_instructions": "exact upgrade/patch steps/commands or 'unknown'",
  "developer_paragraph": "4–6 plain sentences summarizing what to fix/change now",
  "references": ["optional URL 1", "optional URL 2"]
}
"""

SYSTEM_PROMPT = f"""
Rules:
- Choose severity_emoji purely from the impact: 🔥 = high (RCE/credential theft/wormable/lateral movement), ⚠️ = medium, 🧊 = low.
- Prefer and reuse the grounded facts provided in 'hints'. Do NOT contradict them.
- If unknown after reading 'hints', return 'unknown' (or [] for references). Do NOT invent version ranges or URLs.
- Explain the nature of the vulnerability and what exploitation would give to an attacker. 
- Make patch_instructions concrete when possible (package name, firmware page, example commands, or vendor updater UI path).
{DEV_JSON_INSTRUCTIONS}
"""

print("✅ System prompt loaded.")


In [ ]:
# Gather information about the CVES in the list

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

def _nvd_query(cve: str, timeout=12) -> Optional[Dict[str, Any]]:
    try:
        url = "https://services.nvd.nist.gov/rest/json/cves/2.0"
        r = requests.get(url, params={"cveId": cve}, timeout=timeout)
        r.raise_for_status()
        j = r.json()
        items = j.get("vulnerabilities") or []
        return items[0] if items else None
    except Exception:
        return None

def _circl_query(cve: str, timeout=12) -> Optional[Dict[str, Any]]:
    try:
        url = f"https://cve.circl.lu/api/cve/{cve}"
        r = requests.get(url, timeout=timeout)
        if r.status_code != 200:
            return None
        return r.json()
    except Exception:
        return None

def fetch_cve_facts(cve: str) -> Dict[str, Any]:
    print(f"  📥 Fetching CVE facts for {cve} (NVD → CIRCL fallback)...")
    facts = {
        "cve": cve,
        "affected_product": "unknown",
        "affected_versions": "unknown",
        "affected_service": "unknown",
        "cwe": "unknown",
        "description": "unknown",
        "references": []
    }

    nvd = _nvd_query(cve)
    if nvd:
        cve_data = nvd.get("cve", {})
        descs = cve_data.get("descriptions") or []
        if descs:
            facts["description"] = max(descs, key=lambda d: len(d.get("value",""))).get("value","unknown")
        weaks = cve_data.get("weaknesses") or []
        if weaks and weaks[0].get("description"):
            facts["cwe"] = weaks[0]["description"][0].get("value","unknown")
        refs = cve_data.get("references") or []
        facts["references"] = [r.get("url") for r in refs if r.get("url")]
        configs = cve_data.get("configurations") or []
        prods, vers = [], []
        for cfg in configs:
            for node in cfg.get("nodes", []):
                for match in node.get("cpeMatch", []):
                    cpe = match.get("criteria") or ""
                    parts = cpe.split(":")
                    if len(parts) >= 6:
                        vendor = parts[3]; product = parts[4]; version = parts[5]
                        if vendor and product: prods.append(f"{vendor} {product}")
                        if version not in ("*", "-", ""): vers.append(version)
        if prods: facts["affected_product"] = ", ".join(sorted(set(prods)))[:300]
        if vers: facts["affected_versions"] = ", ".join(sorted(set(vers)))[:300]

    circl = _circl_query(cve)
    if circl:
        if facts["description"] == "unknown":
            facts["description"] = circl.get("summary") or "unknown"
        if circl.get("cwe"):
            facts["cwe"] = circl["cwe"]
        if circl.get("references"):
            facts["references"] = list(sorted(set(facts["references"] + circl["references"])))[:12]

    print(f"  ✅ Facts: Product={facts['affected_product']} | Versions={facts['affected_versions']}")
    return facts

def _json_from_text(raw: str) -> Dict[str, Any]:
    raw = raw.strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        import re
        m = re.search(r"\{.*\}", raw, re.S)
        if not m:
            raise RuntimeError(f"Model did not return JSON: {raw[:800]}")
        return json.loads(m.group(0))

def explain_cve_for_developers(cve: str, model: str) -> Dict[str, Any]:
    print(f"\n🔎 Starting analysis for {cve}...")
    facts = fetch_cve_facts(cve)
    user_msg = f"Explain {cve} to a developer. Use these facts:\n{json.dumps(facts)}\nReturn ONLY JSON."

    print(f"  🤖 Sending to OpenAI ({model})...")
    resp = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_msg}
        ]
    )
    print("  📦 Parsing response...")
    parsed = _json_from_text(resp.choices[0].message.content)

    parsed.setdefault("cve", cve)
    parsed.setdefault("severity_emoji", "⚠️")
    parsed.setdefault("references", facts.get("references", []))

    print(f"  ✅ Completed {cve} → {parsed['severity_emoji']}")
    print()
    return parsed

def explain_cves_for_developers(cves, model="gpt-5") -> pd.DataFrame:
    print(f"🚀 Beginning batch analysis for {len(cves)} CVEs...")
    rows = []
    for i, cve in enumerate(cves, start=1):
        print(f"\n=== [{i}/{len(cves)}] {cve} ===")
        try:
            rows.append(explain_cve_for_developers(cve, model))
        except Exception as e:
            print(f"  ❌ Error analyzing {cve}: {e}")
            rows.append({"cve": cve, "severity_emoji": "🧊", "developer_paragraph": f"error: {e}"})
    return pd.DataFrame(rows)


In [ ]:
# Output summary report in markdown

df = explain_cves_for_developers(CVE_LIST, model=MODEL_NAME)

def df_to_markdown_paragraphs(df):
    # Sort by severity: 🔥 > ⚠️ > 🧊, then by CVE
    order = {"🔥": 0, "⚠️": 1, "🧊": 2}
    dfx = df.copy()
    dfx["__ord"] = dfx.get("severity_emoji", "⚠️").map(order).fillna(3)
    dfx = dfx.sort_values(["__ord", "cve"]).drop(columns="__ord", errors="ignore")

    blocks = []
    for _, r in dfx.iterrows():
        cve   = r.get("cve", "").strip()
        emoji = r.get("severity_emoji", "⚠️")

        # Affected line
        affected_bits = []
        for k in ("affected_product", "affected_versions", "affected_service"):
            v = r.get(k, "")
            if isinstance(v, str) and v.strip() and v.strip().lower() != "unknown":
                affected_bits.append(v.strip())
        affected = ", ".join(affected_bits) if affected_bits else "unknown"

        # Core text
        para  = (r.get("developer_paragraph", "") or "").strip()
        patch = (r.get("patch_instructions", "") or "unknown").strip()

        # References (list or string)
        refs = r.get("references") or []
        if isinstance(refs, str):
            refs = [refs]
        refs = [x for x in refs if isinstance(x, str) and x.strip()]
        refs_md = "\n".join(f"- {u}" for u in refs) if refs else "- (none)"

        blocks.append(
            f"{emoji} **{cve}**\n\n"
            f"**Affected:** {affected}\n\n"
            f"{para}\n\n"
            f"**How to patch:**\n\n```\n{patch}\n```\n"
            f"**References:**\n{refs_md}\n\n---\n"
        )
    return "\n".join(blocks)

md = df_to_markdown_paragraphs(df)
display(Markdown(md))

# (Optional) Save to disk
SAVE_REPORT = True
REPORT_PATH = "cve_dev_report.md"
if SAVE_REPORT:
    with open(REPORT_PATH, "w", encoding="utf-8") as f:
        f.write(md)
    print(f"📝 Saved Markdown report → {REPORT_PATH}")

